# First Look at pm-edge Captured Data

Goals: verify data quality on the local archive, get a feel for the universe, look for obvious patterns or oddities at multiple timescales.

In [1]:
# ruff: noqa: F401, I001
import os

os.environ.setdefault(
    "PM_EDGE_LOCAL_FORWARD_INDEX_DIR",
    "/Users/larrymitchell/pm-edge-data/forward_index",
)

from datetime import datetime, timedelta

import pandas as pd

from notebooks.lib.queries import (
    analysis_readiness,
    book_state_quality,
    book_validity,
    capture_freshness,
    data_integrity_checks,
    get_connection,
    hourly_snapshot_volume,
    market_movement,
    metadata_universe,
    multi_timescale_aggregation,
    snapshot_gaps,
    source_breakdown,
    spread_depth_summary,
    trade_overview,
)
from notebooks.lib.plots import (
    plot_book_top_over_time,
    plot_hourly_volume,
    plot_movement_distribution,
    plot_multi_timescale_overlay,
    plot_spread_distribution,
)

con = get_connection()

## Section 1: Capture health overview

In [2]:
df = hourly_snapshot_volume(con)
plot_hourly_volume(df)

In [3]:
source_breakdown(con)

,venue,snapshot_source,count,pct_of_venue
0,kalshi,reconstructed_from_complement,182465,1.000000
1,polymarket,websocket,4348479,0.998309
2,polymarket,rest,7366,0.001691


## Section 2: Data quality verification

In [4]:
book_state_quality(con, venue="polymarket")

,snapshots,avg_bid_levels,avg_ask_levels,pct_with_top_bid,pct_with_top_ask,mean_top_bid,mean_top_ask
0,4355845,3.372551,4.932134,0.809572,0.996265,0.11307,0.096452


In [5]:
book_state_quality(con, venue="kalshi")

,snapshots,avg_bid_levels,avg_ask_levels,pct_with_top_bid,pct_with_top_ask,mean_top_bid,mean_top_ask
0,182465,4.548571,4.510525,0.934837,0.947223,0.50885,0.469699


## Section 3: Integrity, liquidity, and readiness

These checks answer whether the archive is complete enough to analyze. They look for stale capture, missing time buckets, invalid books, shallow liquidity, trade coverage, metadata coverage, duplicate keys, and markets that pass a basic analysis-readiness screen.


In [6]:
capture_freshness(con)


,venue,first_snapshot_utc,latest_snapshot_utc,snapshot_age_minutes,snapshots,distinct_markets,observed_hours,expected_snapshots_at_cadence,pct_expected_snapshots
0,kalshi,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,0.6,182465,277,46.059167,3062013.4,0.059590
1,polymarket,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,0.6,4355845,934,46.059167,10324622.8,0.421889


In [7]:
snapshot_gaps(con).head(25)


,venue,bucket,snapshot_count,venue_median,threshold_count
0,kalshi,2026-05-14 22:20:00-04:00,121,280.0,140.0
1,kalshi,2026-05-14 22:25:00-04:00,40,280.0,140.0
2,kalshi,2026-05-14 22:30:00-04:00,40,280.0,140.0
3,kalshi,2026-05-14 22:35:00-04:00,40,280.0,140.0
4,kalshi,2026-05-14 22:40:00-04:00,40,280.0,140.0
5,kalshi,2026-05-14 22:45:00-04:00,40,280.0,140.0
6,kalshi,2026-05-14 22:50:00-04:00,40,280.0,140.0
7,kalshi,2026-05-14 22:55:00-04:00,38,280.0,140.0
8,kalshi,2026-05-14 23:00:00-04:00,40,280.0,140.0
9,kalshi,2026-05-14 23:05:00-04:00,40,280.0,140.0


In [8]:
book_validity(con)


,venue,snapshots,crossed_books,pct_crossed_books,invalid_price_rows,pct_invalid_price_rows,non_positive_spread_rows,pct_non_positive_spread,pct_empty_bid_levels,pct_empty_ask_levels,pct_with_top_both
0,kalshi,182465,0.0,0.0,0.0,0.0,0.0,0.0,0.065163,0.052777,0.882060
1,polymarket,4355845,0.0,0.0,0.0,0.0,0.0,0.0,0.190428,0.003735,0.806578


In [9]:
spread_depth_summary(con)


,venue,snapshots,spread_p50,spread_p90,spread_p99,spread_mean,avg_bid_depth,avg_ask_depth,avg_depth_imbalance
0,kalshi,182465,0.010,0.05,0.110,0.022541,2.952618e+04,39927.120508,-1.185797e+04
1,polymarket,4355845,0.002,0.02,0.087,0.009094,3.913226e+06,581447.674340,3.227488e+06


In [10]:
display(trade_overview(con))
metadata_universe(con)


,venue,trades,markets_with_trades,contracts,notional,first_trade_utc,latest_trade_utc,duplicate_trade_ids
0,polymarket,67649,489,4.221285e+07,6.159235e+06,2026-05-14 19:47:48.103788-04:00,2026-05-16 17:50:18.075000-04:00,1


,venue,status,metadata_rows,markets,avg_volume_24h,volume_24h_p50,volume_24h_p90,avg_liquidity,liquidity_p50,avg_hours_to_close
0,kalshi,active,4420,410,36272.27860,19893.055000,81278.219000,28784.013977,17913.28000,64.848285
1,polymarket,True,47500,1172,266955.91486,151989.857986,491623.123069,550700.153661,28342.81396,2996.274394


In [11]:
data_integrity_checks(con)


,table_name,check_name,issue_count
0,market_metadata_snapshots,null_critical_fields,0.0
1,order_book_snapshots,duplicate_snapshot_keys,0.0
2,order_book_snapshots,null_critical_fields,0.0
3,order_book_snapshots,schema_versions,1.0
4,trade_events,duplicate_trade_ids,53717.0
5,trade_events,null_critical_fields,0.0


In [12]:
readiness = analysis_readiness(
    con,
    min_snapshots=100,
    min_distinct_top_bids=2,
    max_spread_mean=0.10,
)
display(
    readiness.groupby(["venue", "is_analysis_ready", "exclusion_reason"])
    .size()
    .reset_index(name="markets")
    .sort_values(["venue", "is_analysis_ready", "markets"], ascending=[True, False, False])
)
readiness.head(20)


,venue,is_analysis_ready,exclusion_reason,markets
3,kalshi,True,ready,97
1,kalshi,False,no_top_book_movement,157
0,kalshi,False,incomplete_top_book,21
2,kalshi,False,wide_or_missing_spread,2
7,polymarket,True,ready,399
5,polymarket,False,no_top_book_movement,477
4,polymarket,False,incomplete_top_book,51
6,polymarket,False,wide_or_missing_spread,7


,venue,market_id,snapshots,distinct_top_bids,distinct_top_asks,spread_mean,pct_with_top_both,first_snapshot_utc,latest_snapshot_utc,is_analysis_ready,exclusion_reason
0,polymarket,0x0d6642ddd35287eb369945b9c2b00000b7526d341e3d...,10847,2,1,0.001001,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
1,polymarket,0x0e4a0c937b8934c2475613b6322b3f8edc8dedc24762...,10847,4,4,0.010132,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
2,polymarket,0x2785303a9349d892f464c251c1a39b838a8c535022c5...,10847,2,1,0.006190,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
3,polymarket,0x3733a1b647e7364095736ab0966465d896a84cf3b6bc...,10847,49,49,0.011045,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
4,polymarket,0x411d2f1251d86d9a4f30186c660745c597d18a19737f...,10847,5,6,0.069652,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
5,polymarket,0x4e4a7df876b0c04f0b8b29b9073eddfbaf5c787192da...,10847,6,6,0.001000,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
6,polymarket,0x518a5b030b205706b8ffe6bbad9bd3de59548348e5c0...,10847,21,21,0.008177,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
7,polymarket,0x6a5589de888a72e9e1d711a323ceb1794c9135d4f4a2...,10847,1,2,0.002219,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
8,polymarket,0x789c947a9415600d30d56a4aae88d4111996679b0cae...,10847,6,6,0.001006,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready
9,polymarket,0xafefe6fc14ffb9bd92f2ad0578696345ed3484a566d3...,10847,4,8,0.073590,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 17:50:34.528784-04:00,True,ready


## Section 4: Market movement

In [13]:
movement = market_movement(con)
plot_movement_distribution(movement)

A market with no movement has repeated snapshots but no observed change in `top_bid` or `top_ask`. That can mean a stable liquid market, a stale market, or a capture path that is not receiving live updates. Inspect source breakdown and book depth before interpreting it as market behavior.

## Section 5: Pick a market and look at it

In [14]:
# Pick the market with the most top-of-book changes
most_active = movement.sort_values("distinct_top_bids", ascending=False).iloc[0]
market_id = most_active["market_id"]
print(
    f"Most active market: {market_id} (venue={most_active['venue']}, snapshots={most_active['snapshots']}, distinct_top_bids={most_active['distinct_top_bids']})"
)

aggs = multi_timescale_aggregation(con, market_id)
plot_multi_timescale_overlay(aggs)

Most active market: 0xbede2e0b0fd797304fe8f79fa12c0c2b4e19a2d440af0103c45224fdf4570845 (venue=polymarket, snapshots=1037, distinct_top_bids=183)


## Section 6: Live-watch active Kalshi Eurovision markets

This section watches the most actively-tracked Kalshi Eurovision ranking markets evolve over time. Eurovision 2026 resolves during the broadcast on Saturday evening (May 16). Run this cell repeatedly through the day — after each rsync run, the local archive will have more recent data and the chart will update accordingly.

To refresh the local archive manually before re-running this cell:

```bash
bash ~/ML/pm-edge/deploy/forward_indexer/rsync_to_laptop.sh
```

In [15]:
# Discover all Kalshi Eurovision markets currently in the captured data
eurovision_markets = con.sql("""
    SELECT market_id, COUNT(*) AS snapshots, MIN(timestamp_utc) AS first_seen
    FROM order_book_snapshots
    WHERE venue = 'kalshi' AND market_id LIKE 'KXEUROVISIONRANK%'
    GROUP BY market_id
    ORDER BY snapshots DESC
""").fetchdf()
eurovision_markets

,market_id,snapshots,first_seen
0,KXEUROVISIONRANK-26TOP10-AUS,5154,2026-05-15 19:40:49.129912-04:00
1,KXEUROVISIONRANK-26TOP10-MOL,5153,2026-05-15 19:40:49.129912-04:00
2,KXEUROVISIONRANK-26TOP10-ITA,4761,2026-05-15 19:40:49.129912-04:00
3,KXEUROVISIONRANK-26TOP5-AUS,4379,2026-05-15 19:40:49.129912-04:00
4,KXEUROVISIONRANK-26TOP5-ITA,3104,2026-05-15 19:40:49.129912-04:00
5,KXEUROVISIONRANK-26TOP10-BUL,901,2026-05-16 13:54:19.968984-04:00
6,KXEUROVISIONRANK-26TOP10-ALB,775,2026-05-16 13:54:19.968984-04:00
7,KXEUROVISIONRANK-26TOP3-MOL,760,2026-05-16 13:54:19.968984-04:00
8,KXEUROVISIONRANK-26TOP3-FRA,517,2026-05-16 15:34:34.721376-04:00
9,KXEUROVISIONRANK-26TOP10-ISR,391,2026-05-16 15:34:34.721376-04:00


In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(eurovision_markets) == 0:
    print("No Kalshi Eurovision markets found in the captured data. Nothing to plot.")
else:
    market_ids = eurovision_markets["market_id"].tolist()

    fig = make_subplots(
        rows=len(market_ids),
        cols=1,
        subplot_titles=market_ids,
        shared_xaxes=True,
        vertical_spacing=0.02,
    )

    for i, market_id in enumerate(market_ids, start=1):
        df = con.sql(f"""
            SELECT date_trunc('minute', timestamp_utc) AS minute,
                   AVG(top_bid) AS top_bid,
                   AVG(top_ask) AS top_ask
            FROM order_book_snapshots
            WHERE venue = 'kalshi' AND market_id = '{market_id}'
            GROUP BY minute
            ORDER BY minute
        """).fetchdf()

        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_bid"],
                name="top_bid",
                line={"color": "blue"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_ask"],
                name="top_ask",
                line={"color": "red"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )

    fig.update_layout(
        height=200 * len(market_ids),
        title="Eurovision rank markets — top_bid (blue) and top_ask (red) over time",
        margin={"t": 50, "b": 30, "l": 50, "r": 30},
    )
    fig.show()

## Section 6: Depth-dependent trade investigation

This section investigates trades whose size exceeds the contemporaneous level-1 opposing-side book size. The goal is to identify which markets, categories, and trade-size ranges drive the current depth-walking requirement for simulator v1.


### 6.1 What fraction of trades require depth, by venue and market?

Build a fresh-trade classification table using the latest prior book snapshot within 60 seconds, then rank markets by the fraction of trades larger than level-1 opposing-side size.


In [17]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE depth_trade_classified AS
    WITH trades_with_book AS (
        SELECT
            t.venue,
            t.market_id,
            t.timestamp_utc,
            t.price,
            t.size AS trade_size,
            lower(t.side) AS side,
            b.bid_levels,
            b.ask_levels,
            EXTRACT(EPOCH FROM (t.timestamp_utc - b.timestamp_utc)) AS lag_seconds
        FROM trade_events t
        ASOF LEFT JOIN order_book_snapshots b
            ON t.market_id = b.market_id
            AND t.venue = b.venue
            AND t.timestamp_utc >= b.timestamp_utc
    )
    SELECT
        venue,
        market_id,
        timestamp_utc,
        price,
        trade_size,
        side,
        lag_seconds,
        CASE
            WHEN side = 'buy' AND len(ask_levels) > 0 THEN ask_levels[1].size
            WHEN side = 'sell' AND len(bid_levels) > 0 THEN bid_levels[1].size
            ELSE NULL
        END AS level_1_size,
        CASE
            WHEN side = 'buy' AND len(ask_levels) > 0 AND trade_size > ask_levels[1].size THEN 1
            WHEN side = 'sell' AND len(bid_levels) > 0 AND trade_size > bid_levels[1].size THEN 1
            ELSE 0
        END AS depth_required
    FROM trades_with_book
    WHERE lag_seconds <= 60
""")

depth_overall = con.sql("""
    SELECT
        venue,
        COUNT(*) AS total_trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue
    ORDER BY venue
""").fetchdf()

con.sql("""
    CREATE OR REPLACE TEMP TABLE depth_trades_view AS
    SELECT
        venue,
        market_id,
        COUNT(*) AS total_trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate,
        AVG(trade_size) AS avg_trade_size,
        AVG(level_1_size) AS avg_level_1_size
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue, market_id
    HAVING COUNT(*) >= 20
""")

depth_trades = con.sql("""
    SELECT *
    FROM depth_trades_view
    ORDER BY depth_rate DESC, total_trades DESC
""").fetchdf()

display(depth_overall)
display(depth_trades.head(20))


,venue,total_trades,depth_trades,depth_rate
0,polymarket,63293,6340.0,0.100169


,venue,market_id,total_trades,depth_trades,depth_rate,avg_trade_size,avg_level_1_size
0,polymarket,0x8bb861046d70534248fe10aaef5ca28a91fe4c923db0...,184,112.0,0.608696,276.417105,170.390435
1,polymarket,0x707ab64115b0ca83b65bc5137aaa947a11f56885ed82...,152,88.0,0.578947,205.013616,138.328750
2,polymarket,0xc9615bb82d6535d630ff98e8094060bc767d8b944651...,373,213.0,0.571046,104.327272,153.462225
3,polymarket,0x60ecb823bb667a51ccf9ffcb1cab2c48899cd1641068...,65,33.0,0.507692,91.317824,194.295846
4,polymarket,0xff081c52486b2843da251a6d97187318a578af62f2a1...,250,117.0,0.468000,1410.955983,1127.174560
5,polymarket,0x718b2af2d2b6588ed650d191bf65fe598d70f0b87076...,32,13.0,0.406250,169.308954,248.871875
6,polymarket,0xeb91739b7afb5fdbdb72d0688f593e4ba3887f5e0ada...,44,17.0,0.386364,158.745753,223.007727
7,polymarket,0xb0041709dfdd0d8ae214fd1859b07c899ccd998522d8...,377,141.0,0.374005,635.279669,1000.294934
8,polymarket,0x79e7d16f5c71ab6ff5f805f77045eb5cef7b8f6aa87d...,41,15.0,0.365854,521.016243,982.169024
9,polymarket,0x2baed9d4160afbaf16209d5e2d309393904cd4815b1d...,233,84.0,0.360515,483.738847,899.241459


### 6.2 What does the depth-dependence rate look like as a distribution?

This histogram shows whether depth-dependence is concentrated in a small long tail of markets or spread broadly across the actively traded universe. The dashed reference line marks the full-archive v0 assumption-suite depth dependency rate of `10.32%`.


In [18]:
import plotly.express as px

if len(depth_trades) == 0:
    print("No markets met the minimum trade-count threshold for depth distribution.")
else:
    fig = px.histogram(
        depth_trades,
        x="depth_rate",
        color="venue",
        nbins=40,
        barmode="overlay",
        opacity=0.70,
        title="Distribution of depth-dependence rate by market",
        labels={"depth_rate": "Depth-dependent trade rate", "count": "Markets"},
    )
    fig.add_vline(
        x=0.1032,
        line_dash="dash",
        line_color="black",
        annotation_text="archive avg 10.32%",
    )
    fig.show()


### 6.3 How does depth-dependence correlate with trade size?

Bucket fresh trades by size and compute the depth-dependence rate in each bucket. This identifies where level-1-only simulation starts to understate fills materially.


In [19]:
size_buckets = con.sql("""
    SELECT
        venue,
        CASE
            WHEN trade_size < 10 THEN '00_under_10'
            WHEN trade_size < 100 THEN '01_10_to_100'
            WHEN trade_size < 1000 THEN '02_100_to_1000'
            WHEN trade_size < 10000 THEN '03_1000_to_10000'
            ELSE '04_over_10000'
        END AS size_bucket,
        COUNT(*) AS trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue, size_bucket
    ORDER BY venue, size_bucket
""").fetchdf()

display(size_buckets)

if len(size_buckets) == 0:
    print("No fresh trades with level-1 size available for size-bucket analysis.")
else:
    fig = px.bar(
        size_buckets,
        x="size_bucket",
        y="depth_rate",
        color="venue",
        barmode="group",
        text="trades",
        title="Depth-dependence rate by trade-size bucket",
        labels={
            "size_bucket": "Trade-size bucket",
            "depth_rate": "Depth-dependent trade rate",
            "trades": "Trades",
        },
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()


,venue,size_bucket,trades,depth_trades,depth_rate
0,polymarket,00_under_10,17898,203.0,0.011342
1,polymarket,01_10_to_100,25653,1971.0,0.076833
2,polymarket,02_100_to_1000,14687,2838.0,0.193232
3,polymarket,03_1000_to_10000,4400,1162.0,0.264091
4,polymarket,04_over_10000,655,166.0,0.253435


### 6.4 Can we identify the market categories where depth-dependence concentrates?

Polymarket market metadata does not expose a top-level category column, but `raw_json` includes structured Gamma fields. This section inspects those fields, then groups Polymarket by normalized `feeType` values such as `sports`, `politics`, `crypto`, `culture`, and `finance`. Kalshi tickers still use prefix parsing when trade data exists.


In [ ]:
# Inspect available metadata fields and structured Polymarket category candidates
metadata_sample = con.sql("""
    SELECT
        venue,
        market_id,
        captured_at_utc,
        json_extract_string(raw_json, '$.feeType') AS fee_type,
        json_extract_string(raw_json, '$.question') AS question,
        json_extract_string(raw_json, '$.slug') AS slug,
        json_extract_string(raw_json, '$.groupItemTitle') AS group_item_title
    FROM market_metadata_snapshots
    WHERE venue = 'polymarket'
    LIMIT 5
""").fetchdf()

metadata_fee_types = con.sql("""
    SELECT
        json_extract_string(raw_json, '$.feeType') AS fee_type,
        COUNT(*) AS metadata_rows,
        COUNT(DISTINCT market_id) AS markets
    FROM market_metadata_snapshots
    WHERE venue = 'polymarket'
    GROUP BY fee_type
    ORDER BY markets DESC
""").fetchdf()

display(metadata_sample)
display(metadata_fee_types)


In [20]:
categorized = con.sql("""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            json_extract_string(raw_json, '$.feeType') AS fee_type,
            json_extract_string(raw_json, '$.question') AS question,
            json_extract_string(raw_json, '$.slug') AS slug,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
    ),
    market_depth_with_category AS (
        SELECT
            d.venue,
            d.market_id,
            d.total_trades,
            d.depth_trades,
            d.depth_rate,
            CASE
                WHEN d.venue = 'kalshi' THEN regexp_extract(d.market_id, '^(KX[A-Z]+)', 1)
                WHEN d.venue = 'polymarket' THEN COALESCE(
                    NULLIF(
                        replace(
                            replace(
                                replace(m.fee_type, '_fees_v2', ''),
                                '_fees',
                                ''
                            ),
                            '_prices',
                            ''
                        ),
                        ''
                    ),
                    'polymarket_uncategorized'
                )
                ELSE 'uncategorized'
            END AS category,
            m.fee_type,
            m.question,
            m.slug
        FROM depth_trades_view d
        LEFT JOIN latest_metadata m
            ON d.venue = m.venue
            AND d.market_id = m.market_id
            AND m.rn = 1
    )
    SELECT
        venue,
        category,
        COUNT(*) AS markets,
        SUM(total_trades) AS total_trades,
        SUM(depth_trades) AS depth_trades,
        SUM(depth_trades)::DOUBLE / SUM(total_trades) AS weighted_depth_rate,
        AVG(depth_rate) AS avg_market_depth_rate
    FROM market_depth_with_category
    GROUP BY venue, category
    HAVING SUM(total_trades) >= 100
    ORDER BY weighted_depth_rate DESC, total_trades DESC
""").fetchdf()

categorized


,venue,category,markets,total_trades,depth_trades,weighted_depth_rate,avg_market_depth_rate
0,polymarket,polymarket_unknown,234,62390.0,6259.0,0.100321,0.10342


### Section 6 findings

_Fill this in after running the queries above._

Key observations:
- Overall depth-dependence rate: __%
- Top markets driving depth-dependence: __, __, __
- Distribution shape: [long tail / broad] — _Section 6.2_
- Size threshold for depth-walking importance: trades larger than __ contracts — _Section 6.3_
- Most affected categories: `polymarket_uncategorized`, `culture`, `politics`, `finance`, `sports` by weighted depth rate — _Section 6.4_

V1 simulator implication: [depth walking is essential for category X / depth walking is a long-tail edge case / etc.]


## Section 7: Uncategorized Polymarket markets — what are they?

Section 6.4 showed `polymarket_uncategorized` at 19.50% depth-dependence, higher than any classified category. This section investigates whether those markets share a coherent identity (e.g., a genuine market type we missed) or are a heterogeneous bucket (e.g., metadata-incomplete warm-up artifacts, retired markets, edge cases).

The answer determines whether the categorization logic should be extended or whether the uncategorized bucket is genuinely residual.


In [ ]:
uncategorized_sample = con.sql("""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            raw_json,
            captured_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
        WHERE venue = 'polymarket'
    )
    SELECT market_id, captured_at_utc, raw_json
    FROM latest_metadata
    WHERE rn = 1
      AND (
          json_extract_string(raw_json, '$.feeType') IS NULL
          OR json_extract_string(raw_json, '$.feeType') = ''
      )
    ORDER BY captured_at_utc DESC
    LIMIT 20
""").fetchdf()
uncategorized_sample


In [ ]:
import json

uncategorized_fields = {}
for _, row in uncategorized_sample.iterrows():
    try:
        parsed = json.loads(row["raw_json"])
        for key in parsed:
            uncategorized_fields[key] = uncategorized_fields.get(key, 0) + 1
    except (json.JSONDecodeError, TypeError):
        continue

field_freq = pd.DataFrame(
    [
        {"field": k, "count": v, "pct": v / len(uncategorized_sample) * 100}
        for k, v in uncategorized_fields.items()
    ]
).sort_values("count", ascending=False)
field_freq.head(30)


In [ ]:
samples = []
for _, row in uncategorized_sample.head(10).iterrows():
    try:
        parsed = json.loads(row["raw_json"])
        sample = {
            "market_id_short": row["market_id"][:20] + "...",
            "captured_at": row["captured_at_utc"],
        }
        for field in [
            "question",
            "title",
            "name",
            "description",
            "eventTitle",
            "event_title",
            "outcome",
        ]:
            if field in parsed:
                sample[field] = str(parsed[field])[:150]
                break
        samples.append(sample)
    except (json.JSONDecodeError, TypeError):
        continue

pd.DataFrame(samples)


In [ ]:
uncategorized_depth_check = con.sql("""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            raw_json,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
        WHERE venue = 'polymarket'
    ),
    classified AS (
        SELECT
            d.market_id,
            d.total_trades,
            d.depth_rate,
            d.avg_trade_size,
            d.avg_level_1_size,
            CASE
                WHEN m.raw_json IS NULL THEN 'no_metadata_row'
                WHEN json_extract_string(m.raw_json, '$.feeType') IS NULL THEN 'no_feeType'
                WHEN json_extract_string(m.raw_json, '$.feeType') = '' THEN 'empty_feeType'
                ELSE 'has_feeType'
            END AS classification_status
        FROM depth_trades_view d
        LEFT JOIN latest_metadata m
            ON d.venue = m.venue
            AND d.market_id = m.market_id
            AND m.rn = 1
        WHERE d.venue = 'polymarket'
    )
    SELECT
        classification_status,
        COUNT(*) AS markets,
        SUM(total_trades) AS total_trades,
        AVG(depth_rate) AS avg_depth_rate,
        AVG(avg_trade_size) AS avg_trade_size,
        AVG(avg_level_1_size) AS avg_level_1_size
    FROM classified
    GROUP BY classification_status
    ORDER BY markets DESC
""").fetchdf()
uncategorized_depth_check


### Section 7 findings

Run the queries above and fill in:

- **Uncategorized bucket composition:**
  - no_metadata_row: ___ markets (indexer coverage gap)
  - no_feeType: ___ markets (different market type)
  - empty_feeType: ___ markets (awaiting classification)

- **Coherence assessment:** [coherent — all share characteristic X / scattered — no clear pattern]
- **Recommended action:**
  - If coherent: extend categorization logic to capture them under a new label
  - If scattered: leave as residual, focus depth-walking work on identified high-depth categories instead
  - If indexer coverage gap: file TODO for indexer metadata capture improvement

- **Updated v1 priorities:** [depth-walking for culture markets remains priority / new category X needs attention / no change]


## Findings

_Fill this in after running the notebook._

### Eurovision watch notes

_Re-run Section 6 throughout the day on May 16 to observe how prediction-market top_bid and top_ask evolve as the Eurovision final approaches and resolves. Note: prices below 0.5 mean the market thinks the country is unlikely to make top 5/10; prices climbing toward 1.0 indicate growing confidence. Markets should converge to either 0 or 1 by resolution._